# 00 - Environment setup (run first in every Colab session)
Mounts Drive, clones/pulls the repo, installs the package, seeds RNGs, creates the Drive folders.
Optional: add a Colab secret `GITHUB_TOKEN` (read access) if the repo is private. It is never printed or stored in the repo.

In [ ]:
# Cell 1: configuration (edit here only)
REPO_URL = "https://github.com/notanut/eem-acc-skripsi.git"
REPO_DIR = "/content/eem-acc-skripsi"
DRIVE_ROOT = "/content/drive/MyDrive/Skripsi"   # shared folder added as a shortcut in My Drive
SEED = 42

In [ ]:
# Cell 2: mount Drive, clone or pull repo (token is passed per-command, not saved in .git/config)
import base64, os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

def git(*args, cwd=None):
    cmd = ["git"]
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd += ["-c", f"http.extraheader=Authorization: Basic {basic}"]
    cmd += list(args)
    subprocess.run(cmd, cwd=cwd, check=True)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    git("pull", "--ff-only", cwd=REPO_DIR)
else:
    git("clone", REPO_URL, REPO_DIR)

In [ ]:
# Cell 3: install the package (editable) and dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)

In [ ]:
# Cell 4: seeds + Drive folder structure
os.environ["EEM_ACC_DRIVE_ROOT"] = DRIVE_ROOT
from eem_acc.utils import Paths, set_seed
set_seed(SEED)
paths = Paths().ensure()
print(paths)

In [ ]:
# Cell 5: sanity check - run the test suite (should print "passed")
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR)

In [ ]:
# Cell 6 (optional): sync code to Drive (excludes .git); heavy outputs never go to the repo
import shutil
shutil.copytree(REPO_DIR, paths.code, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(".git", "__pycache__", "*.egg-info", ".pytest_cache"))
shutil.copy(f"{REPO_DIR}/HANDOFF.md", paths.handoff / "HANDOFF.md")   # Drive copy of the handoff
print("code synced to", paths.code, "| HANDOFF.md copied to", paths.handoff)